# 向量数据访问与并行切分

## 概述

第2章我们已经在Add样例里见过多核切分（`offset`）、核内Tile循环、双缓冲（`BUFFER_NUM`）以及手动流水同步的**具体写法**。本节不再重复这些机制，而是把它们统一到一个更高层的视角——**Tiling（数据切分）策略**：数据为什么要切、按什么层级切（多核间 + 核内）、切分时必须遵守哪些硬件约束。理解Tiling策略，是把单个样例推广到任意数据规模、写出正确且高效的Vector算子的关键。

### 学习目标

完成本节后，开发者应能够：

1. 理解Tiling的概念与术语层级（TOTAL_LENGTH / BLOCK_LENGTH / TILE_NUM / TILE_LENGTH）；
2. 掌握两级切分的组合关系：多核间切分与核内Tile切分如何共同决定`tile_length`；
3. 掌握数据切分必须遵守的三条原则（32字节对齐、充分利用UB、多核负载均衡）。

In [ ]:
# 环境初始化
import os, subprocess
env = subprocess.check_output("bash -l -c 'source $ASCEND_TOOLKIT_HOME/set_env.sh && env'", shell=True, text=True)
for line in env.splitlines():
    if "=" in line: os.environ.__setitem__(*line.split("=", 1))
print("环境初始化完成")

---
# 1. 什么是Tiling

Vector算子处理的数据量往往远超单个AI Core片上内存（UB）的容量，无法一次性载入计算。**Tiling（数据切分）** 就是把整体数据按层级切成小块、分块搬运与计算的策略：依据数据shape等信息计算出「每块多大、循环多少次」等参数的过程称为Tiling，所采用的切分算法称为Tiling策略。

pyasc的Tiling分为**两个层级**（第2章Add样例中均已出现，这里给出统一的术语与层级关系）：

<table style="margin-left: 0; margin-right: auto; width: auto; border: 1px solid #ddd;">
  <thead>
    <tr><th align="left">层级</th><th align="left">记号</th><th align="left">含义</th></tr>
  </thead>
  <tbody>
    <tr><td>整体数据</td><td><code>TOTAL_LENGTH</code></td><td>算子输入的总元素个数</td></tr>
    <tr><td>多核间切分</td><td><code>BLOCK_LENGTH</code></td><td>每个核分到的数据长度 = TOTAL_LENGTH ÷ 核数</td></tr>
    <tr><td>核内Tile切分</td><td><code>TILE_LENGTH</code></td><td>每次搬入UB计算的小块长度</td></tr>
  </tbody>
</table>

两级切分的组合关系可以用两行公式概括（与第2章样例完全一致）：

```python
block_length = total_length // USE_CORE_NUM           # 第一级：多核间切分
tile_length  = block_length // TILE_NUM // BUFFER_NUM  # 第二级：核内Tile切分
```

第一级用`asc.get_block_idx()`把互不重叠的GM数据段绑定到各个核（SPMD多核并行，详见第2章2.2）；第二级在单核内用循环逐Tile完成CopyIn→Compute→CopyOut（数据搬运与Tensor切片详见第2章2.4）。本节的重点不是这两段代码本身，而是切分时必须满足的硬件约束——见下一节。

---
# 2. 数据切分的三条原则

Tiling不能随意切分，它受AI Core硬件特性约束。矢量算子做数据切分时应遵循以下三条原则：

<table style="margin-left: 0; margin-right: auto; width: auto; border: 1px solid #ddd;">
  <thead>
    <tr><th align="left">原则</th><th align="left">说明</th></tr>
  </thead>
  <tbody>
    <tr><td><b>1. 32字节对齐</b></td><td>受UB物理限制，UB上的数据存储空间必须保持32字节对齐。输入长度不是32字节整数倍时，需向上对齐到32字节的长度；进行Tiling相关计算时，以32字节为最小单位。</td></tr>
    <tr><td><b>2. 尽量占满UB</b></td><td>AI Core与外部存储交互会产生性能开销，频繁搬运会导致性能瓶颈。应在UB容量范围内尽量增大每块数据，减少从Global Memory搬运数据的次数。</td></tr>
    <tr><td><b>3. 多核负载均衡</b></td><td>AI处理器包含多个AI Core，应把计算尽量均匀分配到多个核上，避免个别核成为瓶颈。</td></tr>
  </tbody>
</table>

> **说明：** 这三条原则决定了切分参数的取值：前两条约束核内`TILE_LENGTH`——既要32字节对齐，又要在UB容量内尽量取大；第三条约束多核间`BLOCK_LENGTH`的分配方式。

样例Add算子的核函数中，两级切分与本节概念的对应关系如下（直接看关键代码段）：

```python
@asc.jit(always_compile=True)
def vadd_kernel(x, y, z, block_length):
    offset = asc.get_block_idx() * block_length       # 各核绑定不重叠的GM数据段（多核间切分 → BLOCK_LENGTH）
    ...
    tile_length = block_length // TILE_NUM // BUFFER_NUM  # 核内切分 → TILE_LENGTH
    for i in range(TILE_NUM * BUFFER_NUM):                # 逐Tile循环处理
        asc.data_copy(x_local[...], x_gm[i * tile_length:], tile_length)
        ...
```

其中`block_length = total_length // USE_CORE_NUM`在Launch函数中算出后传入核函数。样例取`size = 8 * 2048`、启用8核整除，且float32下每块字节数为32的整数倍，天然满足「核间均分 + 32字节对齐」。当总长度不能被核数整除、或分块不是32字节整数倍时，需要额外处理不均匀切分（尾核/尾块场景），属于更复杂的Tiling策略，本课程的入门样例不涉及。

---
# 3. 双缓冲对切分的影响

双缓冲（`BUFFER_NUM = 2`）的原理与流水同步已在第2章2.5详细介绍，本节只关注它对**数据切分**的影响：为了让两块缓冲区能同时驻留在UB中，核内每块Tile的长度要再除以`BUFFER_NUM`：

```python
tile_length = block_length // TILE_NUM // BUFFER_NUM
```

因此UB需要按`tile_length * BUFFER_NUM`的大小申请空间。可以看到，是否启用双缓冲会直接改变`TILE_LENGTH`的取值——这正是Tiling策略需要连同双缓冲一并考虑的原因。

---
# 4. 小结

本节从Tiling策略的视角，统一了Vector算子的数据切分：

- **Tiling概念与术语**：TOTAL_LENGTH →（多核间）BLOCK_LENGTH →（核内）TILE_LENGTH，两级切分共同决定每次计算的数据量；
- **数据切分三原则**：32字节对齐、尽量占满UB、多核负载均衡；
- **双缓冲对切分的影响**：`BUFFER_NUM`使`tile_length`再除以2，UB按`tile_length * BUFFER_NUM`申请空间。

多核`offset`绑定、Tensor切片、双缓冲流水同步的具体机制见第2章2.2 / 2.4 / 2.5，本节不再重复。

---
## 课后练习

**选择题：**

1. Tiling两级切分中，每个核分到的数据长度记为？
   - A. `TILE_LENGTH`
   - B. `BLOCK_LENGTH`
   - C. `TOTAL_LENGTH`
   - D. `BUFFER_NUM`

2. 数据切分三原则中，关于UB存储空间的硬件对齐约束是？
   - A. 必须16字节对齐
   - B. 必须32字节对齐
   - C. 必须64字节对齐
   - D. 无对齐要求

3. `tile_length = block_length // TILE_NUM // BUFFER_NUM`中，再除以`BUFFER_NUM`的原因是？
   - A. 减少核数
   - B. 双缓冲需要两块缓冲区同时驻留UB，每块Tile需相应减半
   - C. 提高数值精度
   - D. 满足32字节对齐

4. 关于Tiling数据切分原则，下列说法**错误**的是？
   - A. 应尽量占满UB，以减少从Global Memory搬运数据的次数
   - B. 应将计算均衡分配到多个AI Core
   - C. UB上数据存储空间需保持32字节对齐
   - D. 每个核处理的数据越少越好，以降低单核压力

**执行以下代码获取答案。**

In [ ]:
!cat ./answer/03.03_answer.txt